# Бинарная модель CatBoost
## 1. Подготовка данных
Читаем готовую кадровую витрину `data/teachers.parquet`: одна строка — педагог в году.
Цель `target=1` — отсутствие в следующем срезе системы, `target=0` — присутствие,
включая переход в другую школу. В 2024 исход неизвестен.

Пока выполняем только подготовку: обучение, Optuna и сохранение модели будут следующими этапами.
Запускать из корня проекта или папки `prototype/`, выбрав Python из `.venv`.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent

df = pd.read_parquet(root / 'data/teachers.parquet')
assert not df.duplicated(['id', 'year']).any()
display(df.groupby('year')['target'].agg(['size', 'count', 'mean']))

,size,count,mean
year,,,
2022,17396,17396,0.149230
2023,17394,17394,0.132172
2024,17933,0,NaN


### Признаки
Все нужные признаки уже есть в parquet. Выбираем их здесь, без импорта кода дашборда.
Идентификаторы и исходы в признаки не входят. Числовые признаки включают бинарные 0/1.

Категориальные столбцы в готовой витрине уже строковые и не имеют пропусков.
Числовые NaN сохраняем: CatBoost умеет с ними работать. Дополнительные признаки,
One-Hot Encoding, масштабирование и заполнение пропусков здесь не нужны.

In [2]:
numeric = [
    'age', 'experience_age_pedagogic_clean', 'experience_age_other_clean',
    'n_subjects', 'n_parallels', 'min_parallel', 'max_parallel',
    'sex', 'class_manager', 'edu_higher', 'edu_pedagogical', 'edu_speciality_match',
    'pk_recent', 'pk_this_year', 'pk_hours_3y', 'retraining',
]
categorical = ['main_position_cat', 'academic_degree_cat', 'category_qual_cat', 'district', 'speciality']
features = numeric + categorical

assert df[categorical].notna().all().all()
display(df[features].dtypes.to_frame('Тип признака'))

,Тип признака
age,int64
experience_age_pedagogic_clean,int64
experience_age_other_clean,int64
n_subjects,int64
n_parallels,int64
min_parallel,int64
max_parallel,int64
sex,int64
class_manager,int64
edu_higher,int64


### Разделение по годам
2022 — обучение, 2023 — итоговый тест. Данные 2024 готовим отдельно для будущего прогноза,
не заполняя неизвестную цель нулями. Ключи `id, year` сохраняем для сопоставления прогнозов.

In [3]:
train = df[df['year'] == 2022]
test = df[df['year'] == 2023]
forecast = df[df['year'] == 2024]

assert not train.empty and not test.empty and not forecast.empty
assert train['target'].isin([0, 1]).all() and test['target'].isin([0, 1]).all()
assert forecast['target'].isna().all()

X_train, y_train = train[features].copy(), train['target'].astype(int)
X_test, y_test = test[features].copy(), test['target'].astype(int)
X_forecast = forecast[features].copy()
forecast_keys = forecast[['id', 'year']].copy()

display(X_train.isna().sum().loc[lambda s: s > 0].to_frame('Пропусков в train'))

,Пропусков в train
edu_speciality_match,16904


### Validation для будущего подбора параметров
Как в HW8, выделяем 20% 2022 года с сохранением доли классов.
`X_fit, y_fit` будут использоваться для обучения при подборе,
`X_valid, y_valid` — для Optuna и early stopping.
2023 не участвует в выборе параметров; 2024 не участвует ни в обучении, ни в проверке.

In [4]:
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42,
)

display(pd.DataFrame({
    'Строк': [len(X_fit), len(X_valid), len(X_test), len(X_forecast)],
    'Признаков': [len(features)] * 4,
    'Доля уходов': [y_fit.mean(), y_valid.mean(), y_test.mean(), pd.NA],
}, index=['Обучение 2022', 'Validation 2022', 'Тест 2023', 'Прогноз 2024']))

,Строк,Признаков,Доля уходов
Обучение 2022,13916,21,0.149253
Validation 2022,3480,21,0.149138
Тест 2023,17394,21,0.132172
Прогноз 2024,17933,21,<NA>


Данные готовы. Для следующего этапа доступны `X_fit`, `X_valid`, `X_train`, `X_test`,
соответствующие цели `y_fit`, `y_valid`, `y_train`, `y_test`, а также `X_forecast` и `forecast_keys`.
Список признаков — `features`, категориальных столбцов для CatBoost — `categorical`.
Обучение, Optuna и сохранение будут реализованы ниже в этом же ноутбуке.
На текущем этапе никакие данные или файлы модели не записываются.

# Модель: прогноз ухода педагога (target)

**Задача:** бинарная классификация — уйдёт педагог в следующем году или останется.  
**Модель:** CatBoost Classifier.  
**Сплит:** временной (обучаемся на ранних годах, валидируем на поздних).  
**Данные:** `teachers.parquet`, зерно `id + year`.

In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc as pr_auc,
)
from pathlib import Path
import joblib
import warnings

warnings.filterwarnings("ignore")

In [6]:
DATA_PATH = Path("data/teachers.parquet")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent

df = pd.read_parquet(root / DATA_PATH)
print(f"Форма: {df.shape}")
print(f"Годы: {sorted(df['year'].unique())}")
print(f"Пропуски target: {df['target'].isna().sum()}")

Форма: (52723, 35)
Годы: [np.int64(2022), np.int64(2023), np.int64(2024)]
Пропуски target: 17933


## Целевая переменная

`target` — бинарный признак: 1 = педагог ушёл, 0 = остался.  
Заполнен только для тех лет, где есть наблюдение следующего года.  
Последний год в данных всегда без target (некуда смотреть вперёд).

In [7]:
df_model = df[df["target"].notna()].copy()
df_model["target"] = df_model["target"].astype(int)

print(f"Строк для обучения: {len(df_model)}")
print(df_model["target"].value_counts(normalize=True))

Строк для обучения: 34790
target
0    0.859299
1    0.140701
Name: proportion, dtype: float64


## Отбор признаков и борьба с утечками (Target Leakage)

Главное правило прогнозного моделирования: **нельзя использовать признаки из будущего**.
В первом запуске мы получили AUC = 1.0 из-за утечки: признак `mobility_status` (статус мобильности) по сути уже содержит ответ на вопрос «ушёл ли педагог». 

Мы должны исключить все признаки, которые:
1. Являются прямым следствием ухода (например, `mobility_status`, `movement_type`).
2. Формируются по итогам года, а не на момент прогноза.

Оставляем только «честные» признаки: демографию, категорию, школу, стаж и нагрузку, зафиксированную на начало периода.

In [8]:
DROP_COLS = [
    "id", "year", "target",
    "school_num",          # сырой идентификатор (модель может переобучиться на конкретные школы)
    "movement_type",       # следствие, а не причина
    "mobility_status",     # Потенциальная утечка. Прямой прокси целевой переменной
]

# Оставляем только те признаки, которые известны на момент прогноза
feature_cols = [
    c for c in df_model.columns
    if c not in DROP_COLS
]

cat_cols = df_model[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Признаков: {len(feature_cols)} (категориальных: {len(cat_cols)}, числовых: {num_cols})")
print(f"Категориальные: {cat_cols}")

# Автоматическая проверка
leak_suspects = [c for c in feature_cols if any(word in c.lower() for word in ['move', 'target', 'status', 'leave', 'quit', 'next'])]
if leak_suspects:
    print(f"⚠️ Подозрительные признаки (возможно они из будущего): {leak_suspects}")
else:
    print("✅ Явных неподходящих признаков в названии не найдено.")

Признаков: 29 (категориальных: 6, числовых: ['age', 'experience_age_pedagogic_clean', 'experience_age_all_clean', 'category_due_clean', 'class_manager', 'sex', 'experience_age_other_clean', 'n_subjects', 'n_parallels', 'min_parallel', 'max_parallel', 'n_subject_parallel', 'school_teachers', 'school_students', 'school_staff_complete', 'students_per_teacher', 'edu_higher', 'edu_pedagogical', 'edu_speciality_match', 'pk_recent', 'pk_hours_3y', 'retraining', 'pk_this_year'])
Категориальные: ['district', 'main_position_cat', 'age_group', 'category_qual_cat', 'speciality', 'academic_degree_cat']
✅ Явных неподходящих признаков в названии не найдено.


## Временной сплит

Случайный сплит запрещён: данные панельные, один педагог встречается в нескольких годах.  
Правильный подход: обучаемся на годах `<= T`, валидируем на годе `T+1`.

Поскольку лет мало (2–3), делаем один срез. Если больше — можно расширить.

In [9]:
years_sorted = sorted(df_model["year"].unique())
split_year = years_sorted[-1]  # последний доступный год с target → валидация

train = df_model[df_model["year"] < split_year]
valid = df_model[df_model["year"] == split_year]

print(f"Train: {len(train)} строк, годы {train['year'].unique()}")
print(f"Valid: {len(valid)} строк, год {valid['year'].unique()}")
print(f"Доля ухода в train: {train['target'].mean():.3f}")
print(f"Доля ухода в valid: {valid['target'].mean():.3f}")

Train: 17396 строк, годы [2022]
Valid: 17394 строк, год [2023]
Доля ухода в train: 0.149
Доля ухода в valid: 0.132


In [10]:
X_train = train[feature_cols]
y_train = train["target"]

X_valid = valid[feature_cols]
y_valid = valid["target"]

cat_indices = [feature_cols.index(c) for c in cat_cols]

train_pool = Pool(X_train, y_train, cat_features=cat_indices)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_indices)

## Параметры CatBoost

- `iterations=1000` — с запасом, остановим через `early_stopping`.
- `learning_rate=0.05` — умеренный, для стабильности.
- `depth=6` — классика для табличных данных.
- `auto_class_weights='Balanced'` — целевая переменная несбалансирована (уходов мало).
- `eval_metric='AUC'` — главная метрика на валидации.

In [11]:
params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3,
    "auto_class_weights": "Balanced",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": 100,
    "early_stopping_rounds": 50,
}

model = CatBoostClassifier(**params)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

0:	test: 0.6859829	best: 0.6859829 (0)	total: 282ms	remaining: 4m 41s
100:	test: 0.7230759	best: 0.7231440 (97)	total: 8.98s	remaining: 1m 19s
200:	test: 0.7228733	best: 0.7242658 (184)	total: 17.8s	remaining: 1m 10s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7242658177
bestIteration = 184

Shrink model to first 185 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, l2_leaf_reg=3, learning_rate=0.05, random_seed=42, verbose=100)

In [12]:
best_iter = model.best_iteration_
best_auc = model.best_score_["validation"]["AUC"]
print(f"Лучшая итерация: {best_iter}")
print(f"AUC на валидации: {best_auc:.4f}")

Лучшая итерация: 184
AUC на валидации: 0.7243


## Оценка качества

Смотрим:
- **AUC-ROC** — общая разделяющая способность.
- **PR-AUC** — важнее при дисбалансе.
- **Classification report** — precision/recall по классам.

In [13]:
y_pred_proba = model.predict_proba(X_valid)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("=== ROC-AUC ===")
print(f"{roc_auc_score(y_valid, y_pred_proba):.4f}")

print("\n=== PR-AUC ===")
prec, rec, _ = precision_recall_curve(y_valid, y_pred_proba)
print(f"{pr_auc(rec, prec):.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_valid, y_pred, target_names=["Остался", "Ушёл"]))

=== ROC-AUC ===
0.7243

=== PR-AUC ===
0.2962

=== Classification Report ===
              precision    recall  f1-score   support

     Остался       0.91      0.81      0.86     15095
        Ушёл       0.28      0.49      0.36      2299

    accuracy                           0.77     17394
   macro avg       0.60      0.65      0.61     17394
weighted avg       0.83      0.77      0.79     17394



In [14]:
importance = model.get_feature_importance(train_pool)
feat_imp = (
    pd.DataFrame({"feature": feature_cols, "importance": importance})
    .sort_values("importance", ascending=False)
)
feat_imp.head(20)

,feature,importance
26,pk_hours_3y,14.576172
3,age_group,7.937098
4,experience_age_pedagogic_clean,7.421174
28,pk_this_year,7.298254
2,age,5.058018
6,category_qual_cat,5.027901
18,school_teachers,5.025074
21,students_per_teacher,4.871538
16,max_parallel,4.600539
8,class_manager,4.103361


## Подбор порога классификации (Threshold Tuning)

Для СППР важна полнота выявления группы риска (Recall). Директору проще просмотреть расширенный список «подозрительных» педагогов и провести с ними профилактические беседы, чем столкнуться с внезапным увольнением.

Сдвинем порог вероятности с дефолтных 0.5 вниз, чтобы добиться Recall ≥ 0.70 для класса «Ушёл».

In [15]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precisions, recalls, thresholds = precision_recall_curve(y_valid, y_pred_proba)

# Ищем порог, где Recall >= 0.70, а Precision максимально возможный
target_recall = 0.70
valid_indices = np.where(recalls[:-1] >= target_recall)[0]

if len(valid_indices) > 0:
    best_idx = valid_indices[np.argmax(precisions[:-1][valid_indices])]
    optimal_threshold = thresholds[best_idx]
    opt_precision = precisions[best_idx]
    opt_recall = recalls[best_idx]
else:
    optimal_threshold = 0.5
    opt_precision, opt_recall = 0.28, 0.49

print(f"Оптимальный порог для Recall >= {target_recall}: {optimal_threshold:.3f}")
print(f"Ожидаемый Recall: {opt_recall:.3f}")
print(f"Ожидаемый Precision: {opt_precision:.3f}")

# Применяем новый порог
y_pred_risk = (y_pred_proba >= optimal_threshold).astype(int)

print("\n=== Новая таблица классификации (Порог = группа риска) ===")
print(classification_report(y_valid, y_pred_risk, target_names=["Вне риска", "Группа риска"]))

Оптимальный порог для Recall >= 0.7: 0.378
Ожидаемый Recall: 0.701
Ожидаемый Precision: 0.223

=== Новая таблица классификации (Порог = группа риска) ===
              precision    recall  f1-score   support

   Вне риска       0.93      0.63      0.75     15095
Группа риска       0.22      0.70      0.34      2299

    accuracy                           0.64     17394
   macro avg       0.58      0.66      0.54     17394
weighted avg       0.84      0.64      0.70     17394



## Профиль группы риска (Инсайты для СППР)

Кого модель чаще всего отправляет в «группу риска»? 
Посмотрим на средние значения ключевых признаков у тех, кого модель считает склонными к уходу (при новом пороге), и сравним с теми, кто вне риска.

In [16]:
valid_df = valid[feature_cols].copy()
valid_df['risk_score'] = y_pred_proba
valid_df['is_risk_group'] = y_pred_risk

profile = valid_df.groupby('is_risk_group').agg(
    Доля_в_выборке=('risk_score', 'size'),
    Средний_возраст=('age', 'mean'),
    Средний_стаж=('experience_age_pedagogic_clean', 'mean'),
    Средние_часы_ПК_3г=('pk_hours_3y', 'mean'),
    Доля_с_ПК_в_этом_году=('pk_this_year', 'mean'),
    Средняя_нагрузка=('students_per_teacher', 'mean')
).T

profile.columns = ['Вне риска', 'Группа риска']
profile

,Вне риска,Группа риска
Доля_в_выборке,10177.000000,7217.000000
Средний_возраст,48.249484,38.953859
Средний_стаж,23.506633,12.413468
Средние_часы_ПК_3г,75.935934,39.258002
Доля_с_ПК_в_этом_году,0.905670,0.575863
Средняя_нагрузка,23.299179,22.819110


## Финализация: Сохранение артефактов для Dash-прототипа

Сохраняем не только саму модель, но и оптимальный порог, список признаков и параметры. 
Вкладка «Прогноз» в Dash будет использовать эти артефакты для расчёта вероятности ухода и формирования списка группы риска для директора школы.

In [17]:
import joblib

# Обновляем артефакты с учётом временного сплита и порога
artifacts = {
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "train_year": 2022,
    "valid_year": 2023,
    "valid_auc": best_auc,
    "optimal_threshold": optimal_threshold,
    "target_recall": opt_recall,
    "target_precision": opt_precision,
}

model.save_model(MODEL_DIR / "catboost_churn.cbm")
joblib.dump(artifacts, MODEL_DIR / "artifacts.pkl")

print("✅ Модель и артефакты сохранены в папке models/")
print(f"   - Модель: catboost_churn.cbm")
print(f"   - Метаданные: artifacts.pkl (порог={optimal_threshold:.3f})")

✅ Модель и артефакты сохранены в папке models/
   - Модель: catboost_churn.cbm
   - Метаданные: artifacts.pkl (порог=0.378)
